In [ ]:
import os
import sys
path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(path)
print(path)

from utility import*

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
from matplotlib.colors import LogNorm
import matplotlib.ticker as ticker
from collections import Counter
import random

In [ ]:
def plot_heatmap(A, N, log=False):
    heatmap = np.zeros((N, N))
    size = 28
    for x, y in A:
        heatmap[x][y] += 1
    
    # Normalize by the maximum value
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    fig, ax = plt.subplots(figsize=(6, 6))  # Set figure size
    im = ax.imshow(heatmap, cmap='gray_r', interpolation='nearest', origin='lower', aspect=3/4)

    # Move colorbar slightly downward
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.2)  # Increase pad slightly to move it down
    cbar.ax.tick_params(labelsize=size-2)

    # Labels and ticks
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xlabel('Destination rack', fontsize=size)
    ax.set_ylabel('Source rack', fontsize=size)
    ax.tick_params(axis='both', which='major', labelsize=size)

    plt.show()

    if log:
        fig, ax = plt.subplots(figsize=(6, 6))
        im = ax.imshow(heatmap, cmap='gray_r', interpolation='nearest', norm=LogNorm(), origin='lower', aspect=3/4)

        cbar = plt.colorbar(im, ax=ax, fraction=0.031, pad=0.05)  # Same pad adjustment for log version
        cbar.ax.tick_params(labelsize=size-3)

        ax.set_yticks([0, 25, 50, 75, 100])
        ax.set_xticks([0, 25, 50, 75, 100])
        ax.set_xlabel('Destination rack', fontsize=size)
        ax.set_ylabel('Source rack', fontsize=size)
        ax.tick_params(axis='both', which='major', labelsize=size)

        plt.show()

def count_distinct_nodes(A):
    distinct_nodes = set()
    for x, y in A:
        distinct_nodes.add(x)
        distinct_nodes.add(y)
    return len(distinct_nodes)

def build_graph(commodities):
    graph = defaultdict(set)
    for u, v in commodities:
        graph[u].add(v)
        graph[v].add(u)
    return graph

def dfs(node, graph, visited, component):
    stack = [node]
    while stack:
        n = stack.pop()
        if n not in visited:
            visited.add(n)
            component.append(n)
            stack.extend(graph[n] - visited)

def find_connected_components(commodities):
    graph = build_graph(commodities)
    visited = set()
    components = []

    for node in graph:
        if node not in visited:
            component = []
            dfs(node, graph, visited, component)
            components.append(sorted(component))

    return components

In [ ]:
def generate_traffic_microsoft(Nrack, Hosts_p_rack, loadfrac0, totaltime, Gbps_rate, Nactive, workload, network, seedValue, commo_list):
    print(f'Running Python function with arguments: {Nrack}, {Hosts_p_rack}, {loadfrac0}, {totaltime}, {Gbps_rate}, {Nactive}, {workload}, {network}, {seedValue}')
    
    filename = f'{network}/{workload}_{100 * loadfrac0:.2f}percLoad_{int(totaltime)}sec_{Nrack}N_{Hosts_p_rack}hpr_{Nrack * Hosts_p_rack}hosts_{Gbps_rate}Gbps_{Nactive:.2f}Nactive_seed={seedValue}.htsim'

    if os.path.exists(filename):
        print(f"File {filename} already exists. Skipping the process.")
        return
    
    np.random.seed(seedValue)
    random.seed(seedValue) 
    
    
    H_active = int(np.ceil(Nrack * Hosts_p_rack * Nactive))
    print(f'H_active = {H_active}')
    
   
    probabilities, srcdst = get_microsoft_probabilities(H_active, Hosts_p_rack, commo_list)
    # Convert the list to a numpy array for further processing
    flowmat1 = get_flow_mat(probabilities, srcdst, workload, Gbps_rate, loadfrac0, H_active, totaltime)

    # Write the flowmat1 to the file in the appropriate format
    write_to_htsim_file(flowmat1, filename)


def get_microsoft_probabilities(H_active, Hosts_p_rack, commo_list):

    Ncons = H_active * (H_active - Hosts_p_rack)  # number of possible connections
    srcdst = np.zeros((Ncons, 2), dtype=int)
    
    # Initialize the probability list
    probabilities = []

    pair_counts = Counter(commo_list)
    total_pairs = len(commo_list)

    cnt = 0
    for a in range(H_active):  # sources
        for b in range(H_active):  # destinations
            if a // Hosts_p_rack != b // Hosts_p_rack:
                # Store the source-destination pair
                srcdst[cnt] = [a, b]

                # Step 2: Determine the frequency of the pair (a, b)
                count = pair_counts.get((a // Hosts_p_rack, b // Hosts_p_rack), 0)  # Get count or 0 if not in commo_list
                p = count / total_pairs if total_pairs > 0 else 0  # Calculate probability based on frequency

                probabilities.append(p)
                cnt += 1

    return probabilities, srcdst


In [ ]:
def get_commo_list_megaswitch(num_nodes, seed_val, theta=0.075, phi=0.40, theta_2=0.34, phi_2=0.30, density=12):
    # num_nodes = 100

    # theta = 0.30
    # phi = 0.05# tor


    # theta_2 = 0.30
    # phi_2 = 0.38# tor
    np.random.seed(seed_val)
    random.seed(seed_val) 
    
    num_commo = density*num_nodes*num_nodes
    num_super_hot_pairs = int(np.ceil(num_nodes * num_nodes * 0.3 / 100))

    all_pairs = [(i, j) for i in range(num_nodes) for j in range(num_nodes) if i != j]
    num_nodes_hot = int(np.ceil(num_nodes * theta))
    num_nodes_medium = int(np.ceil(num_nodes * theta_2))

    hot_nodes = random.sample(range(num_nodes), num_nodes_hot)
    medium_nodes = random.sample([n for n in range(num_nodes) if n not in hot_nodes], num_nodes_medium)
    
    # hot_nodes = [18, 31, 37, 38, 44, 69, 75, 88]
    # medium_nodes = [2, 3, 5, 7, 15, 19, 20, 22, 24, 27, 29, 32, 34, 35, 40, 41, 42, 43, 47, 51, 58, 61, 62, 68, 72, 73, 74, 78, 82, 87, 89, 98, 100, 101]
    print(f"hot nodes = {hot_nodes} {len(hot_nodes)}")
    print(f"medium_nodes' = {medium_nodes} {len(medium_nodes)}")

    super_hot_pairs_candidate = [(i, j) for i in range(num_nodes) for j in range(num_nodes) if i > j and (i in hot_nodes+medium_nodes and j in hot_nodes+medium_nodes)]
    super_hot_pairs = random.sample(super_hot_pairs_candidate, num_super_hot_pairs//2)
    for pair in super_hot_pairs.copy():
        super_hot_pairs.append((pair[1], pair[0]))

    commo_list = []
    prob1 = phi/num_nodes_hot
    prob2 = phi_2/num_nodes_medium
    prob3 = (1 - phi - phi_2) / (num_nodes - num_nodes_hot - num_nodes_medium)

    print(f"{prob1} {prob2} {prob3}")
    while len(commo_list) < num_commo:
        value = random.random()
        
        # if value <= 0.8:
        if value <= 0.5:
            (src, dst) = random.choice(super_hot_pairs)
        else:
            # (src, dst) = random.choice(possible_pairs)
            while True:
                src = random.choices(range(num_nodes), weights=[
                    prob1 if i in hot_nodes else 
                    prob2 if i in medium_nodes else 
                    prob3 for i in range(num_nodes)
                ])[0]
                dst = random.choices(range(num_nodes), weights=[
                    prob1 if i in hot_nodes else 
                    prob2 if i in medium_nodes else 
                    prob3 for i in range(num_nodes)
                ])[0]
                
                if (src in hot_nodes + medium_nodes or dst in hot_nodes + medium_nodes) and (src,dst) not in super_hot_pairs:
                    break
            

        commo_list.append((src, dst))

    # Example: Printing the first few communication pairs
    print(commo_list)
    print(count_distinct_nodes(commo_list))
    print(f"num pair = {len(set(commo_list))} = {len(set(commo_list))/(num_nodes*num_nodes-num_nodes)*100}%")
    plot_heatmap(commo_list, num_nodes, True)

    cc = find_connected_components(commo_list)
    print(cc)
    a = [len(c) for c in cc]
    print(a)
    print(f"super_hot_pairs = {super_hot_pairs}")

    return commo_list

In [ ]:
# def get_commo_list(num_nodes, seed_val, theta=0.075, phi=0.40, theta_2=0.34, phi_2=0.30, density=12):
def get_commo_list(num_nodes, seed_val, theta=0.075, phi=0.40, theta_2=0.34, phi_2=0.30, density=12):
    # num_nodes = 100

    # theta = 0.30
    # phi = 0.05# tor


    # theta_2 = 0.30
    # phi_2 = 0.38# tor
    np.random.seed(seed_val)
    random.seed(seed_val) 
    
    num_commo = density*num_nodes*num_nodes
    num_super_hot_pairs = int(num_nodes * num_nodes * 0.3 / 100)

    all_pairs = [(i, j) for i in range(num_nodes) for j in range(num_nodes) if i != j]
    num_nodes_hot = int(num_nodes * theta)
    num_nodes_medium = int(num_nodes * theta_2)

    hot_nodes = random.sample(range(num_nodes), num_nodes_hot)
    medium_nodes = random.sample([n for n in range(num_nodes) if n not in hot_nodes], num_nodes_medium)
    
    # hot_nodes = [18, 31, 37, 38, 44, 69, 75, 88]
    # medium_nodes = [2, 3, 5, 7, 15, 19, 20, 22, 24, 27, 29, 32, 34, 35, 40, 41, 42, 43, 47, 51, 58, 61, 62, 68, 72, 73, 74, 78, 82, 87, 89, 98, 100, 101]
    print(f"hot nodes = {hot_nodes} {len(hot_nodes)}")
    print(f"medium_nodes' = {medium_nodes} {len(medium_nodes)}")

    super_hot_pairs_candidate = [(i, j) for i in range(num_nodes) for j in range(num_nodes) if i > j and (i in hot_nodes+medium_nodes and j in hot_nodes+medium_nodes)]
    super_hot_pairs = random.sample(super_hot_pairs_candidate, num_super_hot_pairs//2)
    for pair in super_hot_pairs.copy():
        super_hot_pairs.append((pair[1], pair[0]))

    commo_list = []
    prob1 = phi/num_nodes_hot
    prob2 = phi_2/num_nodes_medium
    prob3 = (1 - phi - phi_2) / (num_nodes - num_nodes_hot - num_nodes_medium)

    print(f"{prob1} {prob2} {prob3}")
    while len(commo_list) < num_commo:
        value = random.random()
        
        # if value <= 0.8:
        if value <= 0.5:
            (src, dst) = random.choice(super_hot_pairs)
        else:
            # (src, dst) = random.choice(possible_pairs)
            while True:
                src = random.choices(range(num_nodes), weights=[
                    prob1 if i in hot_nodes else 
                    prob2 if i in medium_nodes else 
                    prob3 for i in range(num_nodes)
                ])[0]
                dst = random.choices(range(num_nodes), weights=[
                    prob1 if i in hot_nodes else 
                    prob2 if i in medium_nodes else 
                    prob3 for i in range(num_nodes)
                ])[0]
                
                if (src in hot_nodes + medium_nodes or dst in hot_nodes + medium_nodes) and (src,dst) not in super_hot_pairs:
                    break
            

        commo_list.append((src, dst))

    # Example: Printing the first few communication pairs
    print(commo_list)
    print(count_distinct_nodes(commo_list))
    print(f"num pair = {len(set(commo_list))} = {len(set(commo_list))/(num_nodes*num_nodes-num_nodes)*100}%")
    plot_heatmap(commo_list, num_nodes, True)

    cc = find_connected_components(commo_list)
    print(cc)
    a = [len(c) for c in cc]
    print(a)
    print(f"super_hot_pairs = {super_hot_pairs}")

    return commo_list

In [ ]:
# commo_list = get_commo_list(num_nodes=108, seed_val=1)

In [ ]:
# network = "clos"
network = "opera"
# network = "megaswitch"

if network == "clos":
    Nrack = 72
    Hosts_p_rack = 9

elif network == "opera":
    Nrack = 108
    Hosts_p_rack = 6

elif network == "megaswitch":
    Nrack = 108
    Hosts_p_rack = 6

    # Nrack = 36
    # Hosts_p_rack = 18

    # Nrack = 25
    # Hosts_p_rack = 26

    # Nrack = 18
    # Hosts_p_rack = 36

workload = "HD"
# time = 10.001
time = 30.001
Gbps_rate = 40
Nactive = 1


# load_set = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12]
load_set = [0.08]
# seed_set = [1, 2, 3, 4, 5]
seed_set = [1, 2, 3, 4, 5]

for seedValue in seed_set:
    if network == "megaswitch":
        commo_list = get_commo_list_megaswitch(num_nodes=Nrack, seed_val=seedValue)
    else:
        commo_list = get_commo_list(num_nodes=Nrack, seed_val=seedValue)
    for load in load_set:
        generate_traffic_microsoft(Nrack, Hosts_p_rack, load, time, Gbps_rate, Nactive, workload, network, seedValue, commo_list)
